
### Методологическое пояснение (Для статьи)

В рамках эксперимента формируются 3 группы векторных индексов для оценки качества поиска:

1.  **Text Retrieval Baselines:** Сравнивается эффективность специализированных текстовых моделей.
    *   **Baseline:** `cointegrated/rubert-tiny2`.
    *   **Target SOTA:** `deepvk/USER-bge-m3` (sota bi-encoder для русского языка).
    *   *Цель:* Показать прирост качества при переходе на VLM-based эмбеддинги.

2.  **Visual Retrieval Ablation:** Сравниваются три архитектуры для поиска схем по описанию.
    *   **SigLIP** (Google), **M-CLIP** (Multilingual), **OpenAI CLIP**.
    *   *Цель:* Выявить лучшую модель для связывания русского технического текста с визуальной топологией схем.

3.  **Negative Control (SigLIP Full Index):**
    *   Создается единый индекс, где и текст учебника (теория), и изображения кодируются одной VLM-моделью (SigLIP).
    *   *Гипотеза:* Данный подход покажет **низкое качество** поиска по теории, так как текстовые энкодеры VLM (Visual-Language Models) имеют короткое контекстное окно и обучены на коротких подписях (captions), а не на абзацах текста. Это доказывает необходимость гибридной архитектуры (отдельный текстовый индекс).
3.  **Jina clip**.


In [1]:
from google.colab import drive
import os
import gc
import torch
import json



drive.mount('/content/drive')

BASE_DIR = "/content/drive/MyDrive/EE_RAG"
DATA_DIR = os.path.join(BASE_DIR, "data")
INDICES_DIR = os.path.join(BASE_DIR, "indices")

os.makedirs(INDICES_DIR, exist_ok=True)

# Устанавливаем библиотеки
!pip install -q faiss-cpu sentence-transformers transformers protobuf

print(f"Данные: {DATA_DIR}")
print(f"Индексы: {INDICES_DIR}")
from PIL import Image
from transformers import AutoProcessor, AutoModel, CLIPProcessor, CLIPModel
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer
from tqdm.notebook import tqdm

def clear_gpu():
    gc.collect()
    torch.cuda.empty_cache()


Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 115.1 MB/s eta 0:00:00
Данные: /content/drive/MyDrive/EE_RAG/data
Индексы: /content/drive/MyDrive/EE_RAG/indices


Текстовый индес ruBert vs USER-bge-m3

In [2]:


# Загрузка данных
with open(os.path.join(DATA_DIR, "theory.json"), 'r', encoding='utf-8') as f:
    theory_data = json.load(f)
texts = [item['text'] for item in theory_data]

print(f"Текстовых чанков: {len(texts)}")

def build_text_index(model_name, filename):
    print(f"\nIndexing Text: {model_name} ...")
    clear_gpu()

    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = SentenceTransformer(model_name, device=device)

    # батч поменьше для BGE
    # USER-bge-m3 требует нормализации для dot product
    embeddings = model.encode(texts, batch_size=16, show_progress_bar=True,
                              convert_to_numpy=True, normalize_embeddings=True)

    d = embeddings.shape[1]
    index = faiss.IndexFlatIP(d)
    index.add(embeddings)

    faiss.write_index(index, os.path.join(INDICES_DIR, filename))
    print(f"Saved: {filename} (Size: {index.ntotal}, Dim: {d})")

    del model
    del embeddings


# 1. Baseline
build_text_index('cointegrated/rubert-tiny2', 'idx_text_rubert.bin')

# 2. SOTA (DeepVK BGE-M3)
build_text_index('deepvk/USER-bge-m3', 'idx_text_user_bge.bin')

Текстовых чанков: 2411

Indexing Text: cointegrated/rubert-tiny2 ...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/963 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/297 [00:00<?, ?B/s]

Batches:   0%|          | 0/151 [00:00<?, ?it/s]

Saved: idx_text_user_bge.bin (Size: 2411, Dim: 1024)


Визуальный индекс

In [ ]:
import torch
import faiss
import numpy as np
import json
import gc
import os
from PIL import Image
from tqdm.notebook import tqdm
from transformers import AutoProcessor, AutoModel, CLIPProcessor, CLIPModel
from sentence_transformers import SentenceTransformer

with open(os.path.join(DATA_DIR, "images.json"), 'r', encoding='utf-8') as f:
    images_data = json.load(f)

valid_files = [img for img in images_data if os.path.exists(img['path'])]
print(f"Картинок для индексации: {len(valid_files)}")

def clear_gpu():
    gc.collect()
    torch.cuda.empty_cache()

def build_vision_index_safe(model_name, model_type, index_filename):
    print(f"Пересборка индекса: {index_filename} ({model_name})")
    clear_gpu()
    device = "cuda" if torch.cuda.is_available() else "cpu"

    try:
        if model_type == 'hf':
            processor = AutoProcessor.from_pretrained(model_name)
            model = AutoModel.from_pretrained(model_name).to(device)
        elif model_type == 'clip':
            processor = CLIPProcessor.from_pretrained(model_name)
            model = CLIPModel.from_pretrained(model_name).to(device)
        elif model_type == 'jina':
            processor = AutoProcessor.from_pretrained(model_name, trust_remote_code=True)
            model = AutoModel.from_pretrained(model_name, trust_remote_code=True).to(device)
    except Exception as e:
        print(f"Ошибка загрузки модели: {e}")
        return

    embeddings = []
    valid_ids = []

    batch_size = 32

    for i in tqdm(range(0, len(valid_files), batch_size)):
        batch = valid_files[i : i + batch_size]
        pil_images = []
        batch_ids_temp = []

        for item in batch:
            try:
                img = Image.open(item['path']).convert("RGB")
                pil_images.append(img)
                batch_ids_temp.append(item['id'])
            except: continue

        if not pil_images: continue

        with torch.no_grad():
            inputs = processor(images=pil_images, return_tensors="pt").to(device)
            vecs = model.get_image_features(**inputs)
            vecs = vecs / vecs.norm(p=2, dim=-1, keepdim=True)
            embeddings.append(vecs.cpu().numpy())

            # Добавляем ID только если инференс прошел успешно
            valid_ids.extend(batch_ids_temp)

    # Сохранение
    if embeddings:
        emb_matrix = np.vstack(embeddings)
        index = faiss.IndexFlatIP(emb_matrix.shape[1])
        index.add(emb_matrix)

        # 1. Save BIN
        faiss.write_index(index, os.path.join(INDICES_DIR, index_filename))

        # 2. Save IDs (JSON)
        ids_path = os.path.join(INDICES_DIR, index_filename.replace(".bin", "_ids.json"))
        with open(ids_path, 'w') as f:
            json.dump(valid_ids, f)

        print(f"УСПЕХ: {index_filename}")
        print(f"   Векторов: {index.ntotal}")
        print(f"   ID записано: {len(valid_ids)}")
    else:
        print("Ошибка: Векторы не созданы!")

build_vision_index_safe("google/siglip-base-patch16-224", "hf", "idx_siglip.bin")
build_vision_index_safe("openai/clip-vit-base-patch32", "clip", "idx_clip_openai.bin")
# Jina вроде была ок, но можно и её обновить для надежности
build_vision_index_safe("jinaai/jina-clip-v1", "jina", "idx_jina.bin")

Картинок для индексации: 2977
Пересборка индекса: idx_siglip.bin (google/siglip-base-patch16-224)


preprocessor_config.json:   0%|          | 0.00/368 [00:00<?, ?B/s]

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


tokenizer_config.json:   0%|          | 0.00/711 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/798k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/409 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/432 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/813M [00:00<?, ?B/s]

  0%|          | 0/94 [00:00<?, ?it/s]

Кодируем И текст, И картинки одной моделью SigLIP, чтобы показать, что для текста это плохо.


In [10]:

def build_full_siglip_index():
    print("Building be index (SigLIP Full)")
    clear_gpu()

    model_name = "google/siglip-base-patch16-224"
    device = "cuda" if torch.cuda.is_available() else "cpu"

    processor = AutoProcessor.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name).to(device)

    combined_embeddings = []
    combined_ids = [] # Храним ID и Тип (text/image)

    # 1. Encode Text (Theory)
    # SigLIP Text Encoder имеет лимит токенов (обычно 64). текст будет обрезан
    print("Encoding Text")
    batch_size = 32
    for i in tqdm(range(0, len(texts), batch_size)):
        batch_txt = texts[i:i+batch_size]
        batch_ids = [theory_data[j]['id'] for j in range(i, min(i+batch_size, len(texts)))]

        with torch.no_grad():
            inputs = processor(text=batch_txt, padding="max_length", truncation=True, return_tensors="pt").to(device)
            vecs = model.get_text_features(**inputs)
            vecs = vecs / vecs.norm(p=2, dim=-1, keepdim=True)
            combined_embeddings.append(vecs.cpu().numpy())

            for bid in batch_ids:
                combined_ids.append({"id": bid, "type": "text"})

    # 2. Encode Images
    # Мы переиспользуем логику, просто для примера загрузим первые 100 картинок, чтобы индекс не раздувать
    # (или загрузим все, если память позволяет)
    print("Encoding Images")
    valid_imgs_subset = valid_imgs  # Берем все

    for i in tqdm(range(0, len(valid_imgs_subset), batch_size)):
        batch = valid_imgs_subset[i:i+batch_size]
        pil_imgs = []
        batch_meta = []
        for item in batch:
            try:
                pil_imgs.append(Image.open(item['path']).convert("RGB"))
                batch_meta.append({"id": item['id'], "type": "image"})
            except: continue

        if not pil_imgs:
            continue

        with torch.no_grad():
            inputs = processor(images=pil_imgs, return_tensors="pt").to(device)
            vecs = model.get_image_features(**inputs)
            vecs = vecs / vecs.norm(p=2, dim=-1, keepdim=True)
            combined_embeddings.append(vecs.cpu().numpy())
            combined_ids.extend(batch_meta)

    if combined_embeddings:
        emb_matrix = np.vstack(combined_embeddings)
        index = faiss.IndexFlatIP(emb_matrix.shape[1])
        index.add(emb_matrix)

        faiss.write_index(index, os.path.join(INDICES_DIR, "idx_bad_siglip_full.bin"))
        with open(os.path.join(INDICES_DIR, "idx_bad_siglip_full_ids.json"), 'w') as f:
            json.dump(combined_ids, f)

        print(f"Negative Control Saved. Size: {index.ntotal}")

build_full_siglip_index()

Building be index (SigLIP Full)
Encoding Text


  0%|          | 0/76 [00:00<?, ?it/s]

Encoding Images


  0%|          | 0/94 [00:00<?, ?it/s]

Negative Control Saved. Size: 5388


Full Jina CLIP Index
 Гипотеза: Jina CLIP (в отличие от SigLIP) должна хорошо искать и по тексту, так как у нее контекст 8к токенов.

In [3]:
def build_jina_full_index():
    print("Jina CLIP Full: Text + Images")
    clear_gpu()

    model_name = "jinaai/jina-clip-v1"
    device = "cuda" if torch.cuda.is_available() else "cpu"

    processor = AutoProcessor.from_pretrained(model_name, trust_remote_code=True)
    model = AutoModel.from_pretrained(model_name, trust_remote_code=True).to(device)

    combined_embeddings = []
    combined_ids = []

    # Загружаем текст заново, на всякий случаай
    with open(os.path.join(DATA_DIR, "theory.json"), 'r', encoding='utf-8') as f:
        theory = json.load(f)
        texts = [t['text'] for t in theory]
        text_ids = [t['id'] for t in theory]

    with open(os.path.join(DATA_DIR, "images.json"), 'r', encoding='utf-8') as f:
        images_data = json.load(f)
        # Фильтруем битые пути
        valid_files = [img for img in images_data if os.path.exists(img['path'])]

    print(f"Encoding {len(texts)}")

    # Уменьшаем batch_size, так как Jina обрабатывает длинные токены
    batch_size_txt = 16

    for i in tqdm(range(0, len(texts), batch_size_txt)):
        batch_txt = texts[i : i + batch_size_txt]
        batch_id = text_ids[i : i + batch_size_txt]

        with torch.no_grad():
            # поддерживает до 8192 токенов.
            # max_length=2048 (для экономии памяти и скорости), достаточно для абзацев учебника.
            inputs = processor(text=batch_txt, padding="max_length", truncation=True, max_length=4096, return_tensors="pt").to(device)

            vecs = model.get_text_features(**inputs)
            vecs = vecs / vecs.norm(p=2, dim=-1, keepdim=True)
            combined_embeddings.append(vecs.cpu().numpy())

            for bid in batch_id:
                combined_ids.append({"id": bid, "type": "text"})

    print("Encoding Images")
    batch_size_img = 32

    for i in tqdm(range(0, len(valid_files), batch_size_img)):
        batch = valid_files[i : i + batch_size_img]
        pil_imgs = []
        batch_id = []

        for item in batch:
            try:
                pil_imgs.append(Image.open(item['path']).convert("RGB"))
                batch_id.append(item['id'])
            except: continue

        if not pil_imgs: continue

        with torch.no_grad():
            inputs = processor(images=pil_imgs, return_tensors="pt").to(device)
            vecs = model.get_image_features(**inputs)
            vecs = vecs / vecs.norm(p=2, dim=-1, keepdim=True)
            combined_embeddings.append(vecs.cpu().numpy())

            for bid in batch_id:
                combined_ids.append({"id": bid, "type": "image"})

    if combined_embeddings:
        emb_matrix = np.vstack(combined_embeddings)
        index = faiss.IndexFlatIP(emb_matrix.shape[1])
        index.add(emb_matrix)

        faiss.write_index(index, os.path.join(INDICES_DIR, "idx_jina_full.bin"))
        with open(os.path.join(INDICES_DIR, "idx_jina_full_ids.json"), 'w') as f:
            json.dump(combined_ids, f)

        print(f"Jina Full Index Saved. Size: {index.ntotal}")

# Запуск
build_jina_full_index()

Jina CLIP Full: Text + Images


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/527 [00:00<?, ?B/s]

processing_clip.py: 0.00B [00:00, ?B/s]

transform.py: 0.00B [00:00, ?B/s]

...

Encoding 2411


  0%|          | 0/151 [00:00<?, ?it/s]

Encoding Images


  0%|          | 0/94 [00:00<?, ?it/s]

Jina Full Index Saved. Size: 5388


In [7]:
clear_gpu()


Все сгенерированные файлы находятся в директории:
`Google Drive/EE_RAG/indices/`


Мы разделили индексы на 3 группы для проведения сравнительных экспериментов (Ablation Study). Все индексы используют метрику **Inner Product (IP)**, которая для нормализованных векторов эквивалентна **Cosine Similarity**.

### Группа А: Текстовые Индексы (Text Retrieval)
*Используются для поиска по `theory.json` (абзацы из учебника).*

| Файл индекса (`.bin`) | Модель-энкодер | Описание | Примечание по маппингу |
| :--- | :--- | :--- | :--- |
| `idx_text_rubert.bin` | `cointegrated/rubert-tiny2` | **Baseline.** Быстрая, легкая модель. | FAISS ID совпадает с индексом в `theory.json` |
| `idx_text_user_bge.bin` | `deepvk/USER-bge-m3` | **Target SOTA.** Мощная модель для русского языка. | FAISS ID совпадает с индексом в `theory.json` |

### Группа Б: Визуальные Индексы (Visual Retrieval)
*Используются для поиска схем по текстовому описанию (Text-to-Image).*

| Файл индекса (`.bin`) | Файл маппинга (`.json`) | Модель | Описание |
| :--- | :--- | :--- | :--- |
| `idx_siglip.bin` | `idx_siglip_ids.json` | `google/siglip...` | **SOTA.** Ожидаем лучшее качество на деталях. |
| `idx_jina.bin` | `idx_jina_ids.json` | `jinaai/jina-clip-v1` | **Language Strong.** Ожидаю лучше качество |
| `idx_clip_openai.bin` | `idx_clip_openai_ids.json` | `openai/clip...` | **Baseline.** Скорее всего, плохо поймет кириллицу. |

### Группа В: Полные Индексы (Full Multimodal Indices)
| Файл | Описание |
| :--- | :--- |
| `idx_bad_siglip_full.bin` | Единый индекс (Текст + Картинки), закодированный SigLIP. Используется, чтобы доказать в статье, что VLM плохо ищет длинные тексты. |
| `dx_jina_full.bin` | Индекс построен моделью Jina CLIP (контекст 8192 токена). Проверяем гипотезу: может ли современная Long-Context VLM качественно искать и теорию, и схемы в одном пространстве |

---


Чтобы использовать эти индексы в следующем ноутбуке (Retrieval), нужно следовать паттерну: **Load -> Encode Query -> Search -> Map**.

Для картинок FAISS ID не может совпадать с номером строки в images.json (так как мы пропускали битые файлы). Нужно использовать ids.json.

```python
# ... загрузка индекса ...
with open(".../idx_siglip_ids.json", "r") as f:
    mapping_ids = json.load(f) # Список реальных ID ['L01_img0', 'L01_img2'...]

with open(".../images.json", "r") as f:
    all_images = json.load(f)
    # Делаем словарь для быстрого поиска: ID -> Объект
    img_lookup = {img['id']: img for img in all_images}

# ... поиск ...
for i, idx in enumerate(I[0]):
    # FAISS ID (int) -> Real ID (str) -> Image Object (dict)
    real_id = mapping_ids[idx]
    image_data = img_lookup[real_id]
    print(f"Найдена картинка: {image_data['path']}")
```
Для группы В

Маппинг-файл содержит не просто строки, а объекты с типом данных, так как в индексе перемешаны текст и картинки.

```python
# Пример для idx_jina_full_ids.json
with open(".../idx_jina_full_ids.json", "r") as f:
    full_mapping = json.load(f)
    # Формат: [{"id": "L01_p1", "type": "text"}, {"id": "L01_img1", "type": "image"}, ...]

# ... поиск ...
for idx in I[0]:
    meta = full_mapping[idx]
    if meta['type'] == 'text':
        print(f"Нашел текст: {meta['id']}")
    else:
        print(f"Нашел картинку: {meta['id']}")
```

Full Jina CLIP Index
 Гипотеза: Jina CLIP (в отличие от SigLIP) должна хорошо искать и по тексту, так как у нее контекст 8к токенов.

> Добавить блок с цитатой


